# Biohub Cell Tracking - Baseline (detect + nearest-neighbor link)
Tracking-by-detection baseline:
1. **Detect** nuclei per timepoint (Gaussian smooth + `peak_local_max`) on the full 3D volume.
2. **Link** consecutive frames by optimal (Hungarian) assignment on physically-scaled centroid distance, gated at `MAX_DIST_UM`.
3. **Write** `submission.csv` (node rows + edge rows) for every dataset in `test/`.
4. **Local score**: reproduce the edge-Jaccard on train samples (they overlap the example test set) to tune params without spending submissions.

GT is sparse (~1 cell/frame) but unmatched predictions are NOT penalized as FP, so we detect densely for high recall. The only over-prediction penalty (`T_true`) is unknown locally; we calibrate it via the leaderboard.

In [1]:
# Setup: zarr>=3 required (geff/OME-Zarr are zarr v3). Internet must be ON for rerun install.
import importlib, subprocess, sys
def pip_install(*pkgs):
    r = subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs], capture_output=True, text=True)
    if r.returncode: print('pip FAILED', pkgs, '\n', r.stderr[-500:])
try:
    import zarr
    if int(zarr.__version__.split('.')[0]) < 3: pip_install('zarr>=3'); importlib.reload(zarr)
except ModuleNotFoundError:
    pip_install('zarr>=3'); import zarr
import numpy as np, pandas as pd, os, glob, time
import scipy.ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max
print('zarr', zarr.__version__)

zarr 3.2.1


In [2]:
# ---------------- CONFIG (tuned via param sweep: thr=0.08 gate=8 -> microJ~0.748) ----------------
VOXEL = np.array([1.625, 0.40625, 0.40625])   # (z,y,x) um per voxel

DET = dict(
    sigma=(0.7, 2.0, 2.0),   # Gaussian blur (z blurred less: coarse axis)
    min_distance=3,          # min voxels between peaks
    threshold_abs=0.08,      # on 0..1 normalized image; LOWER = more detections
    norm_pct=(50.0, 99.8),   # intensity normalization percentiles
    max_peaks=2000,          # safety cap per frame
)
MAX_DIST_UM = 8.0            # linking gate (covers real link motion; max observed 7.81um)

INPUT = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR  = os.path.join(INPUT, 'test')
TRAIN_DIR = os.path.join(INPUT, 'train')
OUT = '/kaggle/working/submission.csv'

In [3]:
# ---------------- IO helpers ----------------
def open_image(zarr_path):
    """Return the (T,Z,Y,X) array of an OME-Zarr sample (largest array = level 0)."""
    g = zarr.open(zarr_path, mode='r')
    if hasattr(g, 'shape'):
        return g
    best, bestsize = None, -1
    def walk(node):
        nonlocal best, bestsize
        try:
            for k in node.keys():
                sub = node[k]
                if hasattr(sub, 'shape'):
                    s = int(np.prod(sub.shape))
                    if s > bestsize: best, bestsize = sub, s
                else: walk(sub)
        except Exception: pass
    walk(g)
    return best

def load_geff(geff_path):
    """Return node_ids, props dict (t,z,y,x arrays), edges (E,2)."""
    g = zarr.open(geff_path, mode='r')
    node_ids = np.asarray(g['nodes/ids'][:])
    props = {}
    for k in list(g['nodes/props'].keys()):
        props[k] = np.asarray(g[f'nodes/props/{k}/values'][:])
    try: edges = np.asarray(g['edges/ids'][:])
    except Exception: edges = np.zeros((0,2), dtype=np.uint64)
    return node_ids, props, edges

In [4]:
# ---------------- Detection ----------------
def detect_centroids(vol, sigma, min_distance, threshold_abs, norm_pct, max_peaks):
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, norm_pct)
    v = np.clip((v - lo) / (hi - lo + 1e-6), 0, 1)
    vs = ndi.gaussian_filter(v, sigma=sigma)
    coords = peak_local_max(vs, min_distance=min_distance, threshold_abs=threshold_abs,
                            exclude_border=False, num_peaks=max_peaks)
    return coords  # (N,3) int voxel coords (z,y,x)

# ---------------- Tracking (detect all frames + NN link) ----------------
def track(arr, det=DET, max_dist_um=MAX_DIST_UM):
    T = arr.shape[0]
    nodes = []            # (node_id, t, z, y, x)
    edges = []            # (src_id, tgt_id)
    nid = 0
    prev_ids, prev_xyz = None, None
    for t in range(T):
        vol = np.asarray(arr[t])
        coords = detect_centroids(vol, **det)
        ids = list(range(nid, nid + len(coords))); nid += len(coords)
        for i, c in zip(ids, coords):
            nodes.append((i, t, int(c[0]), int(c[1]), int(c[2])))
        if prev_xyz is not None and len(prev_xyz) and len(coords):
            D = cdist(prev_xyz * VOXEL, coords * VOXEL)   # scaled um distance
            r, c = linear_sum_assignment(D)
            for i, j in zip(r, c):
                if D[i, j] <= max_dist_um:
                    edges.append((prev_ids[i], ids[j]))
        prev_ids, prev_xyz = ids, coords
    return nodes, edges

In [5]:
# ---------------- Local scorer (base edge Jaccard, per metrics.md) ----------------
# NOTE: applies the matching + edge TP/FP/FN, but NOT the unknown T_true density penalty.
def match_nodes_per_t(gt_nodes, pred_nodes, max_um=7.0):
    """gt_nodes/pred_nodes: list of (id,t,z,y,x). Returns dict gt_id->pred_id (bipartite per t)."""
    from collections import defaultdict
    gt_by_t, pr_by_t = defaultdict(list), defaultdict(list)
    for n in gt_nodes: gt_by_t[n[1]].append(n)
    for n in pred_nodes: pr_by_t[n[1]].append(n)
    g2p = {}
    for t, G in gt_by_t.items():
        P = pr_by_t.get(t, [])
        if not P: continue
        A = np.array([[g[2],g[3],g[4]] for g in G]) * VOXEL
        B = np.array([[p[2],p[3],p[4]] for p in P]) * VOXEL
        D = cdist(A, B)
        r, c = linear_sum_assignment(D)
        for i, j in zip(r, c):
            if D[i, j] <= max_um: g2p[G[i][0]] = P[j][0]
    return g2p

def edge_jaccard(gt_nodes, gt_edges, pred_nodes, pred_edges, max_um=7.0):
    g2p = match_nodes_per_t(gt_nodes, pred_nodes, max_um)
    p2g = {v: k for k, v in g2p.items()}
    pred_set = set(map(tuple, pred_edges))
    gt_set   = set(map(tuple, gt_edges))
    TP = FP = 0
    for a, b in pred_edges:
        if a in p2g and b in p2g:                       # both endpoints matched to GT
            TP += ((p2g[a], p2g[b]) in gt_set)
            FP += ((p2g[a], p2g[b]) not in gt_set)
    FN = sum(1 for u, v in gt_edges
             if not (u in g2p and v in g2p and (g2p[u], g2p[v]) in pred_set))
    denom = TP + FP + FN
    return dict(jaccard=TP/denom if denom else 0.0, TP=TP, FP=FP, FN=FN,
                matched_gt=len(g2p), n_gt_nodes=len(gt_nodes), n_pred_nodes=len(pred_nodes))

In [6]:
# ---------------- Sanity + tuning on a few TRAIN samples ----------------
train_geffs = sorted(glob.glob(os.path.join(TRAIN_DIR, '*.geff')))
print('train samples:', len(train_geffs))

def eval_train_sample(geff_path, det=DET, max_dist_um=MAX_DIST_UM):
    stem = os.path.basename(geff_path)[:-5]
    zarr_path = os.path.join(TRAIN_DIR, stem + '.zarr')
    arr = open_image(zarr_path)
    nid, props, gedges = load_geff(geff_path)
    gt_nodes = [(int(nid[i]), int(props['t'][i]), int(props['z'][i]), int(props['y'][i]), int(props['x'][i]))
                for i in range(len(nid))]
    gt_edges = [(int(s), int(t)) for s, t in gedges]
    p_nodes, p_edges = track(arr, det, max_dist_um)
    return edge_jaccard(gt_nodes, gt_edges, p_nodes, p_edges)

t0 = time.time()
for gp in train_geffs[:5]:
    r = eval_train_sample(gp)
    print(f"{os.path.basename(gp)[:-5]:20s} J={r['jaccard']:.3f} TP={r['TP']} FP={r['FP']} FN={r['FN']} "
          f"matched_gt={r['matched_gt']}/{r['n_gt_nodes']} pred_nodes={r['n_pred_nodes']}")
print('elapsed %.1fs' % (time.time()-t0))

train samples: 199
44b6_0113de3b        J=0.800 TP=40 FP=0 FN=10 matched_gt=52/52 pred_nodes=28294
44b6_0b24845f        J=0.204 TP=10 FP=0 FN=39 matched_gt=19/51 pred_nodes=29716
44b6_0c582fdc        J=0.571 TP=40 FP=0 FN=30 matched_gt=54/71 pred_nodes=25867
44b6_0db75fae        J=0.868 TP=131 FP=0 FN=20 matched_gt=149/157 pred_nodes=15051
44b6_12dfb391        J=0.802 TP=620 FP=0 FN=153 matched_gt=756/788 pred_nodes=55992
elapsed 248.5s


In [7]:
# ---------------- PARAM SWEEP: (threshold_abs, gate) on N train samples ----------------
# Detection is cached per threshold and reused across gates (linking is cheap).
from collections import defaultdict

SWEEP_N       = 10
THRESHOLDS    = [0.18, 0.12, 0.08]
GATES         = [6.0, 8.0, 10.0]
sweep_geffs   = train_geffs[:SWEEP_N]

def detect_all(arr, det):
    return [detect_centroids(np.asarray(arr[t]), **det) for t in range(arr.shape[0])]

def build_graph(coords_list, gate):
    nodes, edges, nid = [], [], 0
    prev_ids, prev_xyz = None, None
    for t, coords in enumerate(coords_list):
        ids = list(range(nid, nid + len(coords))); nid += len(coords)
        for i, c in zip(ids, coords):
            nodes.append((i, t, int(c[0]), int(c[1]), int(c[2])))
        if prev_xyz is not None and len(prev_xyz) and len(coords):
            D = cdist(prev_xyz * VOXEL, coords * VOXEL)
            r, c = linear_sum_assignment(D)
            for i, j in zip(r, c):
                if D[i, j] <= gate: edges.append((prev_ids[i], ids[j]))
        prev_ids, prev_xyz = ids, coords
    return nodes, edges

# preload GT + detections (detections cached per threshold per sample)
gt_cache, det_cache = {}, defaultdict(dict)
t0 = time.time()
for gp in sweep_geffs:
    stem = os.path.basename(gp)[:-5]
    arr = open_image(os.path.join(TRAIN_DIR, stem + '.zarr'))
    nid, props, gedges = load_geff(gp)
    gt_nodes = [(int(nid[i]), int(props['t'][i]), int(props['z'][i]), int(props['y'][i]), int(props['x'][i]))
                for i in range(len(nid))]
    gt_edges = [(int(s), int(t)) for s, t in gedges]
    gt_cache[stem] = (gt_nodes, gt_edges)
    for thr in THRESHOLDS:
        det = {**DET, 'threshold_abs': thr}
        det_cache[stem][thr] = detect_all(arr, det)
    print(f'  detected {stem} ({time.time()-t0:.0f}s)')

# evaluate every (threshold, gate) with micro-averaged Jaccard
print('\n thr   gate |   microJ    TP    FP    FN | pred_nodes  matched%')
for thr in THRESHOLDS:
    for gate in GATES:
        TP=FP=FN=0; npred=0; mg=0; ngt=0
        for stem in gt_cache:
            gt_nodes, gt_edges = gt_cache[stem]
            p_nodes, p_edges = build_graph(det_cache[stem][thr], gate)
            r = edge_jaccard(gt_nodes, gt_edges, p_nodes, p_edges)
            TP+=r['TP']; FP+=r['FP']; FN+=r['FN']; npred+=r['n_pred_nodes']
            mg+=r['matched_gt']; ngt+=r['n_gt_nodes']
        J = TP/(TP+FP+FN) if (TP+FP+FN) else 0
        print(f'{thr:.2f}  {gate:4.1f} | {J:.4f}  {TP:5d} {FP:4d} {FN:5d} | {npred:8d}   {mg/ngt:5.1%}')

  detected 44b6_0113de3b (122s)
  detected 44b6_0b24845f (246s)
  detected 44b6_0c582fdc (366s)
  detected 44b6_0db75fae (486s)
  detected 44b6_12dfb391 (610s)
  detected 44b6_144b256d (747s)
  detected 44b6_1574802b (877s)
  detected 44b6_18ced818 (1019s)
  detected 44b6_1d530831 (1152s)
  detected 44b6_24264f12 (1283s)

 thr   gate |   microJ    TP    FP    FN | pred_nodes  matched%
0.18   6.0 | 0.7065   1382    0   574 |   337825   89.5%
0.18   8.0 | 0.7188   1406    0   550 |   337825   89.5%
0.18  10.0 | 0.7200   1409    1   547 |   337825   89.5%
0.12   6.0 | 0.7290   1426    0   530 |   351384   90.9%
0.12   8.0 | 0.7403   1448    0   508 |   351384   90.9%
0.12  10.0 | 0.7414   1451    1   505 |   351384   90.9%
0.08   6.0 | 0.7362   1440    0   516 |   359077   91.5%
0.08   8.0 | 0.7480   1463    0   493 |   359077   91.5%
0.08  10.0 | 0.7496   1467    1   489 |   359077   91.5%


In [8]:
# ---------------- Generate submission for ALL test datasets ----------------
test_zarrs = sorted(glob.glob(os.path.join(TEST_DIR, '*.zarr')))
print('test datasets:', len(test_zarrs))

rows = []; gid = 0; t0 = time.time()
for zp in test_zarrs:
    ds = os.path.basename(zp)[:-5]
    arr = open_image(zp)
    nodes, edges = track(arr)
    for (nid, t, z, y, x) in nodes:
        rows.append((gid, ds, 'node', nid, t, z, y, x, -1, -1)); gid += 1
    for (s, tg) in edges:
        rows.append((gid, ds, 'edge', -1, -1, -1, -1, -1, s, tg)); gid += 1
    print(f'  {ds}: {len(nodes)} nodes, {len(edges)} edges  ({time.time()-t0:.0f}s)')

cols = ['id','dataset','row_type','node_id','t','z','y','x','source_id','target_id']
sub = pd.DataFrame(rows, columns=cols)
sub.to_csv(OUT, index=False)
print('\nwrote', OUT, sub.shape)
print(sub.head(6).to_string(index=False))

test datasets: 4
  44b6_0113de3b: 28294 nodes, 24363 edges  (49s)
  44b6_0b24845f: 29716 nodes, 21524 edges  (101s)
  6bba_05b6850b: 6427 nodes, 5531 edges  (150s)
  6bba_05db0fb1: 63255 nodes, 54927 edges  (203s)

wrote /kaggle/working/submission.csv (234037, 10)
 id       dataset row_type  node_id  t  z   y   x  source_id  target_id
  0 44b6_0113de3b     node        0  0 41   7 173         -1         -1
  1 44b6_0113de3b     node        1  0 48 101 171         -1         -1
  2 44b6_0113de3b     node        2  0  8  12  83         -1         -1
  3 44b6_0113de3b     node        3  0 20 179  83         -1         -1
  4 44b6_0113de3b     node        4  0 51 137 175         -1         -1
  5 44b6_0113de3b     node        5  0  0  63  58         -1         -1
